In [ ]:
import os
import json
import ast
import math
import time
import random
import zipfile
import shutil
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PROJECT_DIR = Path(".").resolve()
DATA_DIR = PROJECT_DIR / "data"

CSV_PATH = DATA_DIR / "psocr-selected-columns.csv"
ZIP_PATH = DATA_DIR / "images.zip"
IMAGES_DIR = DATA_DIR / "images"

# Full line-level OCR run. No smoke test.
USE_SMOKE_TEST = False
SMOKE_LIMIT_ROWS = 1000

# Main training settings
EPOCHS = 50
BATCH_SIZE = 16
D_MODEL = 256
NUM_LAYERS = 6
RTL = True

RUN_NAME = "smoke" if USE_SMOKE_TEST else "full"
PROC_DIR = DATA_DIR / f"processed_line_{RUN_NAME}"
LINE_CROP_DIR = PROC_DIR / "line_crops"
SPLIT_DIR = PROC_DIR / "splits"

OUTPUT_DIR = PROJECT_DIR / "outputs" / f"vlmamba_line_{RUN_NAME}"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FIGURE_DIR = OUTPUT_DIR / "figures"
PREDICTION_DIR = OUTPUT_DIR / "predictions"

for p in [DATA_DIR, PROC_DIR, LINE_CROP_DIR, SPLIT_DIR, OUTPUT_DIR, CHECKPOINT_DIR, FIGURE_DIR, PREDICTION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("CSV_PATH:", CSV_PATH)
print("ZIP_PATH:", ZIP_PATH)
print("IMAGES_DIR:", IMAGES_DIR)
print("PROC_DIR:", PROC_DIR)
print("LINE_CROP_DIR:", LINE_CROP_DIR)
print("SPLIT_DIR:", SPLIT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("FIGURE_DIR:", FIGURE_DIR)
print("PREDICTION_DIR:", PREDICTION_DIR)

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
# Data extraction section commented/skipped.
# The images and line crops have already been processed previously.
# This avoids extracting images.zip again and saves time.

print("Skipping image extraction because processed line-level data already exists.")
print("Expected original images directory:", IMAGES_DIR)
print("Expected processed line crops directory:", LINE_CROP_DIR)

if LINE_CROP_DIR.exists():
    # We do not count all files here because it can be slow in Jupyter.
    print("Line crop directory found:", LINE_CROP_DIR)
else:
    print("WARNING: Line crop directory not found. Check PROC_DIR and LINE_CROP_DIR paths.")

In [ ]:
# CSV loading section commented/skipped.
# The processed JSONL files and vocabulary will be loaded directly.
# This avoids re-reading the CSV for data processing.

print("Skipping CSV loading for training because processed data already exists.")
print("CSV path, only for reference:", CSV_PATH)
print("Processed data directory:", PROC_DIR)

In [ ]:
def parse_python_list(x):
    """
    The CSV stores lines/tokens like:
    [{'text': '...', 'bbox': [x, y, w, h]}, ...]
    """
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except Exception:
        return []


def normalize_pashto_text(text):
    """
    Minimal safe normalization.
    Keeps Pashto characters and spaces.
    """
    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.replace("\u200c", "")  # zero-width non-joiner
    text = text.replace("\u200f", "")  # RTL mark
    text = text.replace("\ufeff", "")
    text = " ".join(text.split())
    return text.strip()


def safe_crop(img, bbox, pad=2):
    """
    bbox format: [x, y, width, height]
    Kept for reference only. The notebook now loads existing crops.
    """
    x, y, w, h = bbox
    W, H = img.size

    x1 = int(math.floor(x - pad))
    y1 = int(math.floor(y - pad))
    x2 = int(math.ceil(x + w + pad))
    y2 = int(math.ceil(y + h + pad))

    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)

    if x2 <= x1 or y2 <= y1:
        return None

    return img.crop((x1, y1, x2, y2))


def load_jsonl_robust(path):
    """
    Robust JSONL loader.
    Supports normal JSONL and also accidentally concatenated JSON objects.
    """
    path = Path(path)
    rows = []
    decoder = json.JSONDecoder()

    with open(path, "r", encoding="utf-8") as f:
        content = f.read().strip()

    idx = 0
    n = len(content)

    while idx < n:
        while idx < n and content[idx].isspace():
            idx += 1

        if idx >= n:
            break

        obj, end = decoder.raw_decode(content[idx:])
        rows.append(obj)
        idx += end

    return rows


print("Helper functions loaded.")

In [ ]:
# Data processing section commented/skipped.
# Instead of creating line crops again, load the existing processed line-level manifest.

line_manifest_path = PROC_DIR / "line_all.jsonl"

records = []

if line_manifest_path.exists():
    records = load_jsonl_robust(line_manifest_path)
    print("Loaded existing line manifest:", line_manifest_path)
    print("Line records:", len(records))
else:
    # Fallback: load from existing split files if line_all.jsonl is not available.
    print("line_all.jsonl not found. Trying to load records from existing split files...")
    split_paths = [
        SPLIT_DIR / "line_train.jsonl",
        SPLIT_DIR / "line_val.jsonl",
        SPLIT_DIR / "line_test.jsonl",
    ]

    for p in split_paths:
        if p.exists():
            part = load_jsonl_robust(p)
            records.extend(part)
            print("Loaded:", p, "records:", len(part))
        else:
            print("Missing split file:", p)

    print("Total records loaded from splits:", len(records))

assert len(records) > 0, (
    "No processed line-level records found. "
    "Please check PROC_DIR, line_all.jsonl, or split JSONL files."
)

print("Data processing skipped successfully.")
print("Using already processed line-level data from:", PROC_DIR)

In [ ]:
# Preview line crops

sample_records = random.sample(records, min(6, len(records)))

plt.figure(figsize=(14, 8))
for i, rec in enumerate(sample_records):
    img = Image.open(rec["image"]).convert("RGB")
    plt.subplot(3, 2, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(rec["text"][:60], fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Train/validation/test split creation commented/skipped.
# Load the existing processed split files instead of generating new splits.

paths = {
    "train": SPLIT_DIR / "line_train.jsonl",
    "val": SPLIT_DIR / "line_val.jsonl",
    "test": SPLIT_DIR / "line_test.jsonl",
}

for name, p in paths.items():
    assert p.exists(), f"Missing processed split file: {p}"
    n_rows = len(load_jsonl_robust(p))
    print(name, n_rows, p)

print("Existing train/validation/test splits loaded successfully.")

In [ ]:
# Vocabulary building section commented/skipped.
# Load the existing vocabulary instead of rebuilding it.

vocab_path = PROC_DIR / "vocab.json"

assert vocab_path.exists(), f"Missing vocabulary file: {vocab_path}"

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab_preview = json.load(f)

print("Loaded existing vocabulary:", vocab_path)
print("Vocab size:", len(vocab_preview["itos"]))
print("First 30 vocabulary items:")
print(vocab_preview["itos"][:30])

In [ ]:
# Dataset and DataLoader

def load_jsonl(path):
    return load_jsonl_robust(path)


with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = json.load(f)

itos = vocab["itos"]
stoi = vocab["stoi"]
BLANK_ID = vocab["blank_id"]
UNK_ID = vocab["unk_id"]


def encode_text(text):
    text = normalize_pashto_text(text)
    return [stoi.get(ch, UNK_ID) for ch in text]


def decode_ids(ids):
    chars = []
    for i in ids:
        if i == BLANK_ID:
            continue
        if i < 0 or i >= len(itos):
            continue
        ch = itos[i]
        if ch in ["<blank>", "<unk>"]:
            continue
        chars.append(ch)
    return "".join(chars)


def preprocess_image(path, img_height=48, max_width=1024):
    img = Image.open(path).convert("L")

    # Make background approximately white and text dark.
    arr0 = np.array(img)
    if arr0.mean() < 127:
        img = ImageOps.invert(img)

    w, h = img.size
    new_w = max(16, int(round(w * img_height / max(1, h))))
    new_w = min(new_w, max_width)

    resample = Image.Resampling.BICUBIC if hasattr(Image, "Resampling") else Image.BICUBIC
    img = img.resize((new_w, img_height), resample)

    arr = np.array(img).astype(np.float32) / 255.0
    arr = (arr - 0.5) / 0.5  # white ≈ 1, black ≈ -1

    tensor = torch.from_numpy(arr).unsqueeze(0)  # [1, H, W]
    return tensor, new_w


class PashtoLineOCRDataset(Dataset):
    def __init__(self, jsonl_path, img_height=48, max_width=1024):
        self.records = load_jsonl(jsonl_path)
        self.img_height = img_height
        self.max_width = max_width

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        image, width = preprocess_image(
            rec["image"],
            img_height=self.img_height,
            max_width=self.max_width,
        )

        text = normalize_pashto_text(rec["text"])
        label = torch.tensor(encode_text(text), dtype=torch.long)

        return {
            "image": image,
            "width": width,
            "label": label,
            "text": text,
            "path": rec["image"],
        }


def collate_ocr(batch):
    heights = [b["image"].shape[1] for b in batch]
    widths = [b["image"].shape[2] for b in batch]

    H = max(heights)
    W = max(widths)

    # padding value = white background after normalization
    images = torch.full((len(batch), 1, H, W), 1.0, dtype=torch.float32)

    for i, b in enumerate(batch):
        img = b["image"]
        _, h, w = img.shape
        images[i, :, :h, :w] = img

    image_widths = torch.tensor(widths, dtype=torch.long)

    labels = [b["label"] for b in batch]
    label_lengths = torch.tensor([len(x) for x in labels], dtype=torch.long)
    targets = torch.cat(labels, dim=0) if labels else torch.empty(0, dtype=torch.long)

    texts = [b["text"] for b in batch]
    paths_ = [b["path"] for b in batch]

    return {
        "images": images,
        "image_widths": image_widths,
        "targets": targets,
        "target_lengths": label_lengths,
        "texts": texts,
        "paths": paths_,
    }

In [ ]:
# PashtoOCR-VLMamba model

try:
    from mamba_ssm import Mamba
    HAS_MAMBA = True
except Exception as e:
    HAS_MAMBA = False
    print("mamba-ssm not available. Using BiGRU fallback.")
    print("Reason:", str(e))


def sinusoidal_positional_encoding(length, dim, device):
    pe = torch.zeros(length, dim, device=device)
    position = torch.arange(0, length, device=device).unsqueeze(1)
    div_term = torch.exp(
        torch.arange(0, dim, 2, device=device).float() * (-math.log(10000.0) / dim)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    if dim % 2 == 0:
        pe[:, 1::2] = torch.cos(position * div_term)
    else:
        pe[:, 1::2] = torch.cos(position * div_term[:-1])
    return pe.unsqueeze(0)


class BiMambaBlock(nn.Module):
    def __init__(self, d_model=256, dropout=0.1):
        super().__init__()
        self.has_mamba = HAS_MAMBA
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        if self.has_mamba:
            self.fwd = Mamba(
                d_model=d_model,
                d_state=16,
                d_conv=4,
                expand=2,
            )
            self.bwd = Mamba(
                d_model=d_model,
                d_state=16,
                d_conv=4,
                expand=2,
            )
            self.proj = nn.Linear(d_model * 2, d_model)
        else:
            self.gru = nn.GRU(
                input_size=d_model,
                hidden_size=d_model // 2,
                num_layers=1,
                batch_first=True,
                bidirectional=True,
            )
            self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        residual = x
        x_norm = self.norm(x)

        if self.has_mamba:
            y_fwd = self.fwd(x_norm)
            y_bwd = torch.flip(self.bwd(torch.flip(x_norm, dims=[1])), dims=[1])
            y = torch.cat([y_fwd, y_bwd], dim=-1)
            y = self.proj(y)
        else:
            y, _ = self.gru(x_norm)
            y = self.proj(y)

        return residual + self.dropout(y)


class PashtoOCRVLMamba(nn.Module):
    def __init__(
        self,
        vocab_size,
        d_model=256,
        num_layers=6,
        dropout=0.1,
        rtl=True,
    ):
        super().__init__()
        self.rtl = rtl

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(128, d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.SiLU(),

            nn.Conv2d(d_model, d_model, kernel_size=3, padding=1),
            nn.BatchNorm2d(d_model),
            nn.SiLU(),
        )

        self.layers = nn.ModuleList([
            BiMambaBlock(d_model=d_model, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, vocab_size)

    def forward(self, images, image_widths=None):
        """
        images: [B, 1, H, W]
        returns:
            logits: [B, T, vocab_size]
            input_lengths: [B]
        """
        x = self.cnn(images)          # [B, C, H', W']
        x = x.mean(dim=2)            # [B, C, W']
        x = x.permute(0, 2, 1)       # [B, T, C]

        # Pashto is right-to-left, so read visual sequence from right to left.
        if self.rtl:
            x = torch.flip(x, dims=[1])

        T = x.size(1)
        pe = sinusoidal_positional_encoding(T, x.size(-1), x.device)
        x = x + pe

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)
        logits = self.classifier(x)

        if image_widths is None:
            input_lengths = torch.full(
                (images.size(0),),
                T,
                dtype=torch.long,
                device=images.device,
            )
        else:
            # two 2x maxpools reduce width by about 4
            input_lengths = torch.div(
                image_widths.to(images.device),
                4,
                rounding_mode="floor",
            )
            input_lengths = torch.clamp(input_lengths, min=1, max=T)

        return logits, input_lengths


print("Using real mamba-ssm backend:", HAS_MAMBA)

In [ ]:
# Metrics and decoding

def greedy_decode(logits, input_lengths):
    """
    CTC greedy decoding.
    logits: [B, T, V]
    """
    pred_ids = logits.argmax(dim=-1).detach().cpu().numpy()
    input_lengths = input_lengths.detach().cpu().numpy()

    texts = []

    for b in range(pred_ids.shape[0]):
        seq = pred_ids[b, :input_lengths[b]]
        collapsed = []
        prev = None

        for idx in seq:
            idx = int(idx)
            if idx != BLANK_ID and idx != prev:
                collapsed.append(idx)
            prev = idx

        texts.append(decode_ids(collapsed))

    return texts


def edit_distance(a, b):
    """
    Basic Levenshtein distance.
    Works for strings or token lists.
    """
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n

    prev = list(range(m + 1))
    curr = [0] * (m + 1)

    for i in range(1, n + 1):
        curr[0] = i
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            curr[j] = min(
                prev[j] + 1,
                curr[j - 1] + 1,
                prev[j - 1] + cost,
            )
        prev, curr = curr, prev

    return prev[m]


def cer(pred, gold):
    pred = normalize_pashto_text(pred)
    gold = normalize_pashto_text(gold)
    if len(gold) == 0:
        return 0.0 if len(pred) == 0 else 1.0
    return edit_distance(pred, gold) / len(gold)


def wer(pred, gold):
    pred_words = normalize_pashto_text(pred).split()
    gold_words = normalize_pashto_text(gold).split()
    if len(gold_words) == 0:
        return 0.0 if len(pred_words) == 0 else 1.0
    return edit_distance(pred_words, gold_words) / len(gold_words)

In [ ]:
# Training configuration

IMG_HEIGHT = 48
MAX_WIDTH = 1024

# Full line-level OCR training
BATCH_SIZE = 16
EPOCHS = 50

D_MODEL = 256
NUM_LAYERS = 6

LR = 3e-4
WEIGHT_DECAY = 1e-4
AMP = True
RTL = True

train_ds = PashtoLineOCRDataset(paths["train"], img_height=IMG_HEIGHT, max_width=MAX_WIDTH)
val_ds = PashtoLineOCRDataset(paths["val"], img_height=IMG_HEIGHT, max_width=MAX_WIDTH)
test_ds = PashtoLineOCRDataset(paths["test"], img_height=IMG_HEIGHT, max_width=MAX_WIDTH)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_ocr,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_ocr,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_ocr,
)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))
print("Vocab size:", len(itos))
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Image height:", IMG_HEIGHT)
print("Max width:", MAX_WIDTH)
print("D_MODEL:", D_MODEL)
print("NUM_LAYERS:", NUM_LAYERS)

In [ ]:
# Initialize model

model = PashtoOCRVLMamba(
    vocab_size=len(itos),
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    dropout=0.1,
    rtl=RTL,
).to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(model.__class__.__name__)
print("Parameters:", f"{num_params:,}")
print("Trainable:", f"{trainable_params:,}")
print("Device:", DEVICE)

In [ ]:
# Train and validate with training-time measurement

criterion = nn.CTCLoss(blank=BLANK_ID, zero_infinity=True)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, EPOCHS),
)

scaler = torch.cuda.amp.GradScaler(enabled=(AMP and DEVICE.type == "cuda"))

best_val_cer = float("inf")
history = {
    "train_loss": [],
    "val_loss": [],
    "val_cer": [],
    "val_wer": [],
    "epoch_time_sec": [],
}

best_ckpt_path = CHECKPOINT_DIR / "best_line_ocr.pt"
last_ckpt_path = CHECKPOINT_DIR / "last_line_ocr.pt"
training_time_path = OUTPUT_DIR / "training_time_summary.json"


def format_seconds(seconds):
    seconds = float(seconds)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:05.2f}"


def run_validation(model, loader):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    all_pred = []
    all_gold = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Validation", leave=False):
            images = batch["images"].to(DEVICE)
            image_widths = batch["image_widths"].to(DEVICE)
            targets = batch["targets"].to(DEVICE)
            target_lengths = batch["target_lengths"].to(DEVICE)

            logits, input_lengths = model(images, image_widths)

            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)
            loss = criterion(
                log_probs,
                targets,
                input_lengths,
                target_lengths,
            )

            total_loss += loss.item()
            total_batches += 1

            preds = greedy_decode(logits, input_lengths)
            all_pred.extend(preds)
            all_gold.extend(batch["texts"])

    mean_loss = total_loss / max(1, total_batches)
    mean_cer = float(np.mean([cer(p, g) for p, g in zip(all_pred, all_gold)]))
    mean_wer = float(np.mean([wer(p, g) for p, g in zip(all_pred, all_gold)]))

    return mean_loss, mean_cer, mean_wer


training_start_time = time.perf_counter()
print("Training started.")
print("Epochs:", EPOCHS)
print("Start time:", time.strftime("%Y-%m-%d %H:%M:%S"))

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.perf_counter()

    model.train()

    train_loss = 0.0
    train_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for batch in pbar:
        images = batch["images"].to(DEVICE)
        image_widths = batch["image_widths"].to(DEVICE)
        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(AMP and DEVICE.type == "cuda")):
            logits, input_lengths = model(images, image_widths)

            log_probs = F.log_softmax(logits, dim=-1).transpose(0, 1)

            loss = criterion(
                log_probs,
                targets,
                input_lengths,
                target_lengths,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_batches += 1

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    avg_train_loss = train_loss / max(1, train_batches)
    val_loss, val_cer_score, val_wer_score = run_validation(model, val_loader)

    epoch_time_sec = time.perf_counter() - epoch_start_time

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_cer"].append(val_cer_score)
    history["val_wer"].append(val_wer_score)
    history["epoch_time_sec"].append(epoch_time_sec)

    total_elapsed_sec = time.perf_counter() - training_start_time
    avg_epoch_time_sec = total_elapsed_sec / epoch
    remaining_epochs = EPOCHS - epoch
    estimated_remaining_sec = avg_epoch_time_sec * remaining_epochs

    print(
        f"Epoch {epoch}: "
        f"train_loss={avg_train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"val_CER={val_cer_score:.4f}, "
        f"val_WER={val_wer_score:.4f}, "
        f"epoch_time={format_seconds(epoch_time_sec)}, "
        f"elapsed={format_seconds(total_elapsed_sec)}, "
        f"estimated_remaining={format_seconds(estimated_remaining_sec)}"
    )

    ckpt = {
        "model": model.state_dict(),
        "vocab": vocab,
        "config": {
            "img_height": IMG_HEIGHT,
            "max_width": MAX_WIDTH,
            "d_model": D_MODEL,
            "num_layers": NUM_LAYERS,
            "rtl": RTL,
            "vocab_size": len(itos),
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
        },
        "history": history,
        "epoch": epoch,
        "val_cer": val_cer_score,
        "val_wer": val_wer_score,
        "elapsed_training_time_sec": total_elapsed_sec,
    }

    torch.save(ckpt, last_ckpt_path)

    if val_cer_score < best_val_cer:
        best_val_cer = val_cer_score
        torch.save(ckpt, best_ckpt_path)
        print("Saved best checkpoint:", best_ckpt_path)

total_training_time_sec = time.perf_counter() - training_start_time

training_summary = {
    "total_training_time_sec": total_training_time_sec,
    "total_training_time_hms": format_seconds(total_training_time_sec),
    "epochs": EPOCHS,
    "average_epoch_time_sec": total_training_time_sec / max(1, EPOCHS),
    "average_epoch_time_hms": format_seconds(total_training_time_sec / max(1, EPOCHS)),
    "best_val_cer": best_val_cer,
    "end_time": time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(training_time_path, "w", encoding="utf-8") as f:
    json.dump(training_summary, f, ensure_ascii=False, indent=2)

print("Training finished.")
print("Best validation CER:", best_val_cer)
print("Total training time:", format_seconds(total_training_time_sec))
print("Average epoch time:", training_summary["average_epoch_time_hms"])
print("Training-time summary saved:", training_time_path)

In [ ]:
# Plot training curves in one figure and save at 700 DPI

from pathlib import Path
import matplotlib.pyplot as plt

SAVE_DIR = Path("outputs/figures")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
axes[0].plot(history["train_loss"], label="Train Loss", linewidth=2)
axes[0].plot(history["val_loss"], label="Validation Loss", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("PashtoOCR-VLMamba Loss")
axes[0].legend()

# CER/WER curve
axes[1].plot(history["val_cer"], label="Validation CER", linewidth=2)
axes[1].plot(history["val_wer"], label="Validation WER", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Error Rate")
axes[1].set_title("PashtoOCR-VLMamba CER/WER")
axes[1].legend()

plt.tight_layout()

save_path = SAVE_DIR / "pashtoocr_vlmamba_training_curves_700dpi.png"
plt.savefig(save_path, dpi=700, bbox_inches="tight")

plt.show()

print("Figure saved at:", save_path)

In [ ]:
# Evaluate best model on test set

ckpt = torch.load(best_ckpt_path, map_location=DEVICE)

model.load_state_dict(ckpt["model"])
model.to(DEVICE)
model.eval()

test_loss, test_cer, test_wer = run_validation(model, test_loader)

print("Test Loss:", test_loss)
print("Test CER:", test_cer)
print("Test WER:", test_wer)

In [ ]:
# Show predictions

model.eval()

batch = next(iter(test_loader))

images = batch["images"].to(DEVICE)
image_widths = batch["image_widths"].to(DEVICE)

with torch.no_grad():
    logits, input_lengths = model(images, image_widths)
    preds = greedy_decode(logits, input_lengths)

for i in range(min(10, len(preds))):
    print("=" * 80)
    print("IMAGE:", batch["paths"][i])
    print("GOLD:", batch["texts"][i])
    print("PRED:", preds[i])
    print("CER :", cer(preds[i], batch["texts"][i]))

In [ ]:
# Inference on one image

def predict_one_image(image_path):
    model.eval()

    img_tensor, width = preprocess_image(
        image_path,
        img_height=IMG_HEIGHT,
        max_width=MAX_WIDTH,
    )

    images = img_tensor.unsqueeze(0).to(DEVICE)
    widths = torch.tensor([width], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        logits, input_lengths = model(images, widths)
        pred = greedy_decode(logits, input_lengths)[0]

    return pred


sample_path = test_records[0]["image"]

print("Image:", sample_path)
print("Gold:", test_records[0]["text"])
print("Prediction:", predict_one_image(sample_path))

img = Image.open(sample_path).convert("RGB")
plt.figure(figsize=(12, 3))
plt.imshow(img)
plt.axis("off")
plt.show()